# Jupyter Notebook: Exploratory Data Analysis (EDA) on Raw GNSS Telemetry

* **Author:** Yahav Alkoby
* **Institution:** Holon Institute of Technology (HIT)
* **Course:** Introduction to Data Science — Assignment 1: Exploratory Data Analysis (EDA)
* **Dataset:** Google Smartphone Decimeter Challenge (`device_gnss_p4xl_train.csv`)

---

### Section 1 — Setup & Environment Configuration
In this initial cell, we set up the Python environment for exploratory data analysis.
* **Libraries:** We import `pandas` and `numpy` for data manipulation, `matplotlib` and `seaborn` for visualization, and `scipy.stats` for statistics.
* **Visual Style:** Seaborn's theme ensures consistent, publication-ready plots.
* **Warning Suppression:** Non-critical visual warnings are filtered out to maintain a clean notebook output.


In [2]:
# ==============================================================================
# SECTION 1: ENVIRONMENT SETUP & CONFIGURATION
# ==============================================================================

import os
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

# Configure visualization settings
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12
warnings.filterwarnings("ignore")

print("Section 1: Environment setup completed successfully.")


Section 1: Environment setup completed successfully.


---
### Section 3 — Data Loading & Meta-Analysis
This section loads the raw GNSS telemetry dataset and computes high-level metadata:
1. **File Size Metrics:** Calculates memory consumption on disk (in MB).
2. **Dimensionality:** Checks the total count of rows and features.
3. **Column Structural Inspection:** Evaluates the data type, non-null count, and missingness percentage.


In [1]:
# ==============================================================================
# SECTION 3: META-ANALYSIS & DATASTRUCTURE INSPECTION
# ==============================================================================

file_path = "device_gnss_train_p4xl.csv"

if not os.path.exists(file_path):
    file_path = "device_gnss_p4xl_train.csv"

file_size_mb = os.path.getsize(file_path) / (1024 * 1024) if os.path.exists(file_path) else 0.0
gnss_df = pd.read_csv(file_path)

print("=" * 70)
print("SECTION 3: META-ANALYSIS & DATASTRUCTURE INSPECTION")
print("=" * 70)
print(f"File Name: {os.path.basename(file_path)}")
print(f"File Size: {file_size_mb:.2f} MB")
print(f"Dimensions: {gnss_df.shape[0]:,} rows x {gnss_df.shape[1]} columns")
print("-" * 70)

meta_summary = pd.DataFrame({
    "Data_Type": gnss_df.dtypes,
    "Non_Null_Count": gnss_df.notnull().sum(),
    "Null_Count": gnss_df.isnull().sum(),
    "Null_Percentage": (gnss_df.isnull().sum() / len(gnss_df)) * 100,
})
display(meta_summary)


NameError: name 'os' is not defined

---
### Section 4 — Data Quality & Integrity Validation
Data quality validation checks for domain anomalies, integrity flaws, and zero-variance features:
* **Physical Domain Violations:** Satellite Elevation Angle must physically lie between 0 and 90 degrees. Carrier-to-Noise Density ($C/N_0$) typically falls between 10 and 60 dB-Hz. Values outside these indicate sensor errors.


In [ ]:
# ==============================================================================
# SECTION 4: DATA QUALITY & INTEGRITY
# ==============================================================================

print("=" * 70)
print("SECTION 4: DATA QUALITY & INTEGRITY")
print("=" * 70)

missing_series = gnss_df.isnull().sum()
print("4.1 Missing Values (Columns with >0 missing entries):")
print(missing_series[missing_series > 0])

full_duplicates = gnss_df.duplicated().sum()
print(f"\n4.2 Full Row Duplicates: {full_duplicates:,}")

invalid_elevation = gnss_df[gnss_df["SvElevationDegrees"] < 0] if "SvElevationDegrees" in gnss_df.columns else None
suspicious_cn0 = gnss_df[(gnss_df["Cn0DbHz"] < 10) | (gnss_df["Cn0DbHz"] > 60)] if "Cn0DbHz" in gnss_df.columns else None

print(f"\n4.3 Suspicious Elevation Entries (<0 deg): {0 if invalid_elevation is None else len(invalid_elevation):,}")
print(f"4.3 Suspicious C/N0 Signal Entries (<10 or >60 dB-Hz): {0 if suspicious_cn0 is None else len(suspicious_cn0):,}")

unique_counts = gnss_df.nunique()
print("\n4.4 Zero Variance Columns (Single unique value):")
print(unique_counts[unique_counts <= 1])
print("=" * 70)


---
### Section 5 — Univariate Analysis & 3 Outlier Detection Methods
We evaluate the main signal quality metric: Carrier-to-Noise Density Ratio (`Cn0DbHz`) using Standard Z-Score, Tukey's IQR Method, and Modified Z-Score based on Median Absolute Deviation (MAD).


In [ ]:
# ==============================================================================
# SECTION 5: UNIVARIATE ANALYSIS & 3 OUTLIER METHODS
# ==============================================================================

column = "Cn0DbHz"
series = gnss_df[column].dropna()

mean_val, median_val, std_val = series.mean(), series.median(), series.std()
mad_val = (series - median_val).abs().mean()
iqr_val = series.quantile(0.75) - series.quantile(0.25)
skewness = series.skew()

print("=" * 70)
print(f"SECTION 5: UNIVARIATE STATISTICS FOR COLUMN '{column}'")
print("=" * 70)
print(f"Mean:   {mean_val:.2f}    | Median: {median_val:.2f}  | Std Dev: {std_val:.2f}")
print(f"MAD:    {mad_val:.2f}    | IQR:    {iqr_val:.2f}  | Skewness: {skewness:.2f}")
print(f"Min:    {series.min():.2f}    | Max:    {series.max():.2f}")
print("-" * 70)

# Outlier Methods
z_scores = np.abs(stats.zscore(series))
outliers_z = series[z_scores > 3]

q1, q3 = series.quantile(0.25), series.quantile(0.75)
outliers_iqr = series[(series < (q1 - 1.5 * iqr_val)) | (series > (q3 + 1.5 * iqr_val))]

median_absolute_deviation = np.median(np.abs(series - median_val))
mod_z_scores = 0.6745 * np.abs(series - median_val) / (median_absolute_deviation + 1e-9)
outliers_mad = series[mod_z_scores > 3.5]

print(f"Method 1 (Z-Score > 3):        {len(outliers_z):,}")
print(f"Method 2 (IQR Rule):           {len(outliers_iqr):,}")
print(f"Method 3 (Modified MAD > 3.5): {len(outliers_mad):,}")
print("=" * 70)


---
### Section 6.1 & 6.2 — Correlation Analysis & Cramér's V
Evaluates multi-variable associations using numerical correlation measures (Pearson, Spearman, Kendall) and one categorical-categorical metric (Cramér's V) via a contingency table.


In [ ]:
# ==============================================================================
# SECTION 6.1 & 6.2: CORRELATIONS & INTER-VARIABLE RELATIONSHIPS
# ==============================================================================

num_cols = ["SvElevationDegrees", "Cn0DbHz", "RawPseudorangeMeters"]
num_df = gnss_df[num_cols].dropna()

print("=" * 70)
print("SECTION 6: CORRELATIONS & INTER-VARIABLE RELATIONSHIPS")
print("=" * 70)
print("6.1 Pearson Correlation Matrix (Linear):")
display(num_df.corr(method="pearson").round(3))

print("\n6.1 Spearman Rank Correlation Matrix (Monotonic):")
display(num_df.corr(method="spearman").round(3))

df_binned = gnss_df.copy()
df_binned["Signal_Strength_Bin"] = pd.qcut(df_binned["Cn0DbHz"], q=4, labels=["Weak", "Moderate", "Strong", "Excellent"])
contingency = pd.crosstab(df_binned["SignalType"], df_binned["Signal_Strength_Bin"])

chi2 = stats.chi2_contingency(contingency)[0]
n = contingency.sum().sum()
phi2 = chi2 / n
r, k = contingency.shape
v_stat = np.sqrt(phi2 / min(k - 1, r - 1))

print("-" * 70)
print(f"6.2 Cramér's V Correlation (SignalType vs Binned C/N0): {v_stat:.3f}")
print("=" * 70)


---
### Section 6.3 — Visualization Suite
A collection of charts mapping signal variance, tracking distributions, and elevation dynamics across the dataset.


In [ ]:
# 1. Scatterplot & 2. Histogram
clean_gps = gnss_df[(gnss_df["ConstellationType"] == 1) & (gnss_df["SignalType"] == "GPS_L1_CA")].copy()

plt.figure(figsize=(10, 5))
sns.scatterplot(data=clean_gps, x="SvElevationDegrees", y="Cn0DbHz", alpha=0.4, color="dodgerblue")
plt.title("Scatterplot: Signal Strength (C/N0) vs. Elevation Angle (GPS L1)", fontsize=14)
plt.xlabel("Elevation Angle [Degrees]")
plt.ylabel("C/N0 [dB-Hz]")
plt.show()

plt.figure(figsize=(9, 5))
sns.histplot(clean_gps["Cn0DbHz"], bins=35, kde=True, color="teal")
plt.title("Histogram: Distribution of Carrier-to-Noise Density (C/N0)", fontsize=14)
plt.show()


In [ ]:
# 3. Satellite Tracking Frequencies
plt.figure(figsize=(10, 5))
top_svids = clean_gps["Svid"].value_counts().head(10)
sns.barplot(x=top_svids.index, y=top_svids.values, palette="viridis", hue=top_svids.index, legend=False)
plt.title("Barchart: Top 10 Most Frequently Tracked GPS Satellites (Svid)", fontsize=14)
plt.show()

# 4. Boxplot across Constellations
plt.figure(figsize=(10, 5))
sns.boxplot(data=gnss_df, x="ConstellationType", y="Cn0DbHz", palette="Blues", hue="ConstellationType", legend=False)
plt.title("Boxplot: Signal Strength Distribution Across GNSS Constellations", fontsize=14)
plt.show()


In [ ]:
# 5. Correlation Heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(gnss_df[["SvElevationDegrees", "Cn0DbHz", "RawPseudorangeMeters"]].corr(method="spearman"), annot=True, cmap="coolwarm", fmt=".3f")
plt.title("Heatmap: Spearman Rank Correlation Matrix", fontsize=14)
plt.show()


---
### Section 7 & 9 — Index Structure & Domain Feature Engineering
Creates a domain-specific classifier feature: `Is_Reliable_LOS` (Line-of-Sight). A signal is classified as a reliable direct line-of-sight signal if its elevation is $\ge 40^\circ$ and $C/N_0 \ge 37\text{ dB-Hz}$.


In [ ]:
# ==============================================================================
# SECTION 7: INDEX STRUCTURE & SECTION 9: FEATURE ENGINEERING
# ==============================================================================

print("=" * 70)
print("SECTION 7: INDEX STRUCTURE ANALYSIS")
print(f"Is DataFrame Index Unique? -> {gnss_df.index.is_unique}")
print(f"Is DataFrame Index Monotonically Increasing? -> {gnss_df.index.is_monotonic_increasing}")

gnss_df["Is_Reliable_LOS"] = np.where(
    (gnss_df["SvElevationDegrees"] >= 40) & (gnss_df["Cn0DbHz"] >= 37),
    1,
    0,
)

los_percentage = (gnss_df["Is_Reliable_LOS"].mean()) * 100
print("=" * 70)
print("SECTION 9 (BONUS): DOMAIN FEATURE ENGINEERING")
print(f"Reliable Line-of-Sight (LOS) Measurements in Dataset: {los_percentage:.2f}%")
print("=" * 70)
